# Genetic Algorithm Strategies for Enhanced Pattern Detection

This notebook searches actual OHLCV price history for X-A-B-C-D harmonic-pattern candidates. It extracts alternating local swing points, uses a genetic algorithm to select five-point sequences, and scores each sequence against explicit ratio bands.

**Interpretation:** matches are heuristic candidates. The supplied data has no labels, so the score is ratio fit, not accuracy, predictive performance, or a trading signal. The ratio bands and swing filter are configurable assumptions.

Set `DATASET_FILE` to `dataset.csv` or `dataset1.csv`. Both files are expected to sit beside this notebook.

In [ ]:
from pathlib import Path
import random
import pandas as pd
import plotly.graph_objects as go

# Set this to dataset1.csv to run the same workflow against the second included file.
DATASET_FILE = Path("dataset.csv")
MIN_SWING_PCT = 0.02
POPULATION_SIZE = 120
GENERATIONS = 60
MUTATION_RATE = 0.25
RANDOM_SEED = 7
MAX_GAP = 7  # Odd gaps preserve alternating high/low swing directions.

# Heuristic ratio bands used by this prototype. They are configuration choices,
# not learned from labels in the supplied data.
PATTERN_BANDS = {
    "Gartley": {"B/XA": (0.568, 0.668), "C/AB": (0.382, 0.886), "CD/BC": (1.130, 1.618), "D/XA": (0.736, 0.836)},
    "Bat": {"B/XA": (0.382, 0.500), "C/AB": (0.382, 0.886), "CD/BC": (1.618, 2.618), "D/XA": (0.836, 0.936)},
    "Butterfly": {"B/XA": (0.750, 0.820), "C/AB": (0.382, 0.886), "CD/BC": (1.618, 2.240), "D/XA": (1.190, 1.350)},
    "Crab": {"B/XA": (0.382, 0.618), "C/AB": (0.382, 0.886), "CD/BC": (2.240, 3.618), "D/XA": (1.520, 1.720)},
}


def load_prices(path):
    data = pd.read_csv(path)
    required = {"Date", "Open", "High", "Low", "Close"}
    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")
    data = data.drop_duplicates().copy()
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
    for column in ("Open", "High", "Low", "Close"):
        data[column] = pd.to_numeric(
            data[column].astype(str).str.replace(r"[$,]", "", regex=True),
            errors="coerce",
        )
    data = data.dropna(subset=["Date", "Open", "High", "Low", "Close"])
    data = data.sort_values("Date").reset_index(drop=True)
    if len(data) < 20:
        raise ValueError("At least 20 valid OHLC rows are needed for swing analysis.")
    return data


def find_swings(close_prices, order=5, min_swing_pct=MIN_SWING_PCT):
    """Return alternating local high/low pivots after a minimum-move filter."""
    prices = list(map(float, close_prices))
    candidates = []
    for i in range(order, len(prices) - order):
        window = prices[i - order:i + order + 1]
        value = prices[i]
        if value == max(window) and value > min(window):
            candidates.append({"index": i, "kind": "High", "price": value})
        elif value == min(window) and value < max(window):
            candidates.append({"index": i, "kind": "Low", "price": value})

    swings = []
    for point in candidates:
        if not swings:
            swings.append(point)
            continue
        previous = swings[-1]
        if point["kind"] == previous["kind"]:
            more_extreme = (point["price"] > previous["price"] if point["kind"] == "High"
                            else point["price"] < previous["price"])
            if more_extreme:
                swings[-1] = point
            continue
        move = abs(point["price"] - previous["price"]) / max(abs(previous["price"]), 1e-12)
        if move >= min_swing_pct:
            swings.append(point)
    return swings


def decode_genome(genome):
    start, *gaps = genome
    indices = [start]
    for gap in gaps:
        indices.append(indices[-1] + gap)
    return tuple(indices)


def repair_genome(genes, swing_count):
    start = max(0, min(int(genes[0]), swing_count - 5))
    allowed_gaps = tuple(range(1, MAX_GAP + 1, 2))
    gaps = []
    for value in genes[1:5]:
        gaps.append(min(allowed_gaps, key=lambda allowed: abs(allowed - int(value))))
    while start + sum(gaps) > swing_count - 1:
        largest = max(range(4), key=lambda i: gaps[i])
        if gaps[largest] > 1:
            gaps[largest] -= 2
        elif start > 0:
            start -= 1
        else:
            break
    return [start, *gaps]


def random_genome(rng, swing_count):
    genes = [rng.randint(0, swing_count - 5)]
    genes.extend(rng.choice(tuple(range(1, MAX_GAP + 1, 2))) for _ in range(4))
    return repair_genome(genes, swing_count)


def measure_ratios(points):
    x, a, b, c, d = [point["price"] for point in points]
    xa, ab, bc, cd = abs(a - x), abs(b - a), abs(c - b), abs(d - c)
    if min(xa, ab, bc, cd) <= 1e-12:
        return None
    return {
        "B/XA": ab / xa,
        "C/AB": bc / ab,
        "CD/BC": cd / bc,
        "D/XA": abs(d - a) / xa,
    }


def score_pattern(ratios, bands):
    normalized_errors = []
    inside_all_bands = True
    for ratio_name, (low, high) in bands.items():
        value = ratios[ratio_name]
        half_width = max((high - low) / 2, 1e-9)
        center = (low + high) / 2
        normalized_errors.append(min(abs(value - center) / half_width, 2.0))
        inside_all_bands &= low <= value <= high
    score = 100 * (1 - sum(normalized_errors) / len(normalized_errors))
    return score, inside_all_bands


def evaluate_genome(genome, swings):
    points = [swings[index] for index in decode_genome(genome)]
    ratios = measure_ratios(points)
    if ratios is None:
        return -100.0, None
    best = None
    for name, bands in PATTERN_BANDS.items():
        score, valid = score_pattern(ratios, bands)
        fitness = score + (50 if valid else 0)
        if best is None or fitness > best["fitness"]:
            best = {"name": name, "score": score, "valid": valid,
                    "fitness": fitness, "ratios": ratios, "points": points}
    return best["fitness"], best


def run_genetic_search(swings, seed=RANDOM_SEED):
    if len(swings) < 5:
        return [], []
    rng = random.Random(seed)
    population = [random_genome(rng, len(swings)) for _ in range(POPULATION_SIZE)]
    hall = {}
    history = []
    elite_count = max(2, POPULATION_SIZE // 20)

    for generation in range(GENERATIONS):
        ranked = []
        for genome in population:
            fitness, result = evaluate_genome(genome, swings)
            ranked.append((fitness, genome, result))
            if result and result["valid"]:
                indices = decode_genome(genome)
                key = (indices, result["name"])
                hall[key] = result
        ranked.sort(key=lambda item: item[0], reverse=True)
        history.append({"generation": generation + 1, "best_fitness": ranked[0][0]})
        next_population = [list(item[1]) for item in ranked[:elite_count]]

        def tournament():
            contestants = rng.sample(ranked, k=min(3, len(ranked)))
            return list(max(contestants, key=lambda item: item[0])[1])

        while len(next_population) < POPULATION_SIZE:
            parent_a, parent_b = tournament(), tournament()
            if rng.random() < 0.8:
                cut = rng.randint(1, 4)
                child_a = parent_a[:cut] + parent_b[cut:]
                child_b = parent_b[:cut] + parent_a[cut:]
            else:
                child_a, child_b = parent_a[:], parent_b[:]
            for child in (child_a, child_b):
                for gene_index in range(5):
                    if rng.random() < MUTATION_RATE:
                        if gene_index == 0:
                            child[gene_index] = rng.randint(0, len(swings) - 5)
                        else:
                            child[gene_index] = rng.choice(tuple(range(1, MAX_GAP + 1, 2)))
                next_population.append(repair_genome(child, len(swings)))
                if len(next_population) == POPULATION_SIZE:
                    break
        population = next_population

    detections = []
    for (indices, name), result in hall.items():
        points = result["points"]
        if points[-1]["kind"] != points[0]["kind"]:
            continue
        row = {
            "Pattern": name,
            "Direction": "Bullish candidate" if points[-1]["kind"] == "Low" else "Bearish candidate",
            "Score": round(result["score"], 1),
            "X date": data.loc[points[0]["index"], "Date"].date(),
            "A date": data.loc[points[1]["index"], "Date"].date(),
            "B date": data.loc[points[2]["index"], "Date"].date(),
            "C date": data.loc[points[3]["index"], "Date"].date(),
            "D date": data.loc[points[4]["index"], "Date"].date(),
            "D price": points[4]["price"],
            "B/XA": round(result["ratios"]["B/XA"], 3),
            "C/AB": round(result["ratios"]["C/AB"], 3),
            "CD/BC": round(result["ratios"]["CD/BC"], 3),
            "D/XA": round(result["ratios"]["D/XA"], 3),
            "_indices": indices,
        }
        detections.append(row)
    detections.sort(key=lambda row: row["Score"], reverse=True)
    return detections, history


data = load_prices(DATASET_FILE)
swings = find_swings(data["Close"])
detections, fitness_history = run_genetic_search(swings)

print(f"Dataset: {DATASET_FILE} | rows: {len(data)} | swing points: {len(swings)}")
print(f"Candidate patterns satisfying all configured ratio bands: {len(detections)}")

if detections:
    results = pd.DataFrame(detections).drop(columns="_indices")
    display(results.head(20))
else:
    print("No candidate met every configured ratio band in this run.")

# Plot the price series and the actual swing points for the top candidates.
fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=data["Date"], open=data["Open"], high=data["High"],
    low=data["Low"], close=data["Close"], name="OHLC"
))
for rank, detection in enumerate(detections[:10], start=1):
    selected = [data.loc[index] for index in detection["_indices"]]
    fig.add_trace(go.Scatter(
        x=[row["Date"] for row in selected],
        y=[row["Close"] for row in selected],
        mode="lines+markers",
        name=f"{rank}. {detection['Pattern']} ({detection['Score']:.1f})",
    ))
fig.update_layout(
    title="Genetic search: harmonic-pattern candidates",
    xaxis_title="Date", yaxis_title="Price", xaxis_rangeslider_visible=False,
    legend_title="Top candidates",
)
fig.show()

fitness_chart = go.Figure(go.Scatter(
    x=[row["generation"] for row in fitness_history],
    y=[row["best_fitness"] for row in fitness_history],
    mode="lines", name="Best population fitness",
))
fitness_chart.update_layout(
    title="Genetic search progress", xaxis_title="Generation",
    yaxis_title="Fitness (ratio fit plus valid-band bonus)",
)
fitness_chart.show()


## Method and limitations

- Swing points come from centered local extrema followed by a minimum percentage-move filter. This uses future bars in each centered window, so it is suitable for historical exploration, not a real-time alert without redesign.
- The genetic algorithm searches combinations of alternating pivots. Fitness rewards closeness to the configured ratios and adds a bonus when all four ratios fall inside the selected pattern bands.
- Candidate ranking is not validated against expert labels. Add labeled examples and out-of-sample evaluation before making claims about detection accuracy or predictive value.
- Dataset source and redistribution terms were not present in the supplied archive; confirm them before redistributing the CSV files.